In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## Dataset Paths and Output Directories

In [6]:
real_videos_dir = "/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences"
manipulated_videos_dir = "/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences"

output_real_dir = "/kaggle/working/frames/real"
output_manipulated_dir = "/kaggle/working/frames/manipulated"

# Make sure directories exist
os.makedirs(output_real_dir, exist_ok=True)
os.makedirs(output_manipulated_dir, exist_ok=True)

if os.path.exists(manipulated_videos_dir):
    print(f"Found manipulated folder. It contains {len(os.listdir(manipulated_videos_dir))} items.")
else:
    print("Warning: The path to the manipulated videos is still incorrect.")

## Frame Extraction from Videos

In [8]:
def extract_frames_from_videos(videos_dir, output_dir, label, max_videos=100):
    video_files = [f for f in os.listdir(videos_dir) if f.endswith(('.mp4', '.avi', '.mov', '.mkv'))]
    video_files = video_files[:max_videos]

    for video_file in video_files:
        video_path = os.path.join(videos_dir, video_file)
        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        success, image = cap.read()

        while success:
            if frame_count % int(cap.get(cv2.CAP_PROP_FPS)) == 0:
                frame_filename = f"{label}_{video_file}_frame{frame_count // int(cap.get(cv2.CAP_PROP_FPS))}.jpg"
                frame_path = os.path.join(output_dir, frame_filename)
                cv2.imwrite(frame_path, image)
            success, image = cap.read()
            frame_count += 1

        cap.release()

# Run the extraction for 100 videos each
print("Extracting real video frames...")
extract_frames_from_videos(real_videos_dir, output_real_dir, "real", max_videos=100)

print("Extracting manipulated video frames...")
extract_frames_from_videos(manipulated_videos_dir, output_manipulated_dir, "manipulated", max_videos=100)

print("Frame extraction completed!")

print("Real frames:", len(os.listdir('/kaggle/working/frames/real')))
print("Manipulated frames:", len(os.listdir('/kaggle/working/frames/manipulated')))

## Data Preparation and Augmentation

In [ ]:
dataset_dir = "/kaggle/working/frames"
batch_size = 32
img_size = (224, 224)

# 1. Corrected Keras Augmentation Pipeline (Removed vertical flip)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.055),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomContrast(factor=0.2),
    layers.RandomBrightness(factor=0.2),
])

# 2. Load Datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

# 3. Split Validation/Test
val_batches = tf.data.experimental.cardinality(val_test_ds)
val_ds = val_test_ds.take(val_batches // 2)
test_ds = val_test_ds.skip(val_batches // 2)

# 4. Corrected Prepare Function
def prepare(ds, augment=False):
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)

    # Cast to float32, but LEAVE values in [0, 255].
    # EfficientNetV2 handles the rest internally.
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(buffer_size=tf.data.AUTOTUNE)

train_ds = prepare(train_ds, augment=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

## Phase 1: Feature Extraction with a Pretrained EfficientNetV2B0

In [ ]:
# 1. Use Pretrained Weights (Transfer Learning)
# Using ImageNet weights gives the model a massive head start in feature extraction
base_model = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights='imagenet', # CHANGED: From None to 'imagenet'
    input_shape=(224, 224, 3),
    pooling='avg'
)

# Freeze the base model for the first phase of training
base_model.trainable = False

# 3. Build a More Robust Classification Head
inputs = tf.keras.Input(shape=(224, 224, 3))

# Pass inputs to the base model (training=False ensures BatchNorm layers don't update during freezing)
x = base_model(inputs, training=False)

# Add Dropout and a Dense layer to help the model learn complex combinations of features safely
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(2, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# 4. Define Optimizer and Compile
# Use a higher learning rate for the newly initialized Dense layers
optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Smarter Callbacks
# ReduceLROnPlateau is generally more adaptive than a strict step decay
callbacks_list = [
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint('./best_tf_model.keras', monitor='val_accuracy', save_best_only=True)
]

# 6. Address Class Imbalance (Optional but recommended based on your report)
# Your report shows 364 Manipulated vs 313 Real. This tells the model to pay slightly more attention to Real.
class_weights = {
    0: 1.16,  # Assuming 0 is 'Real'
    1: 1.00   # Assuming 1 is 'Manipulated'
}

# 7. Train Phase 1: Train only the custom head
print("Starting Phase 1 Training (Feature Extraction)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15, # Shorter epoch run for the frozen model
    class_weight=class_weights, # Apply the weights here
    callbacks=callbacks_list
)

## Phase 2: Fine-Tuning the Entire Model

In [ ]:
# Train Phase 2: Fine-Tuning (Unfreeze and train everything at a low learning rate)
print("Starting Phase 2 Training (Fine-Tuning)...")
base_model.trainable = True

# Recompile with a VERY LOW learning rate so we don't wreck the pretrained weights
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks_list
)

print(f"Best Fine-Tuned Validation Accuracy: {max(history_fine.history['val_accuracy']) * 100:.2f}%")

## Evaluation on Test Set

In [ ]:
model.load_weights('./best_tf_model.keras')

print("Generating predictions on the test dataset in a single pass...")

all_labels = []
all_predictions = []

# Single-pass loop: Guarantees predictions and labels are perfectly aligned
for images, labels in test_ds:
    # Predict on the current batch
    preds = model.predict(images, verbose=0)

    # Store predictions and true labels for this exact batch
    all_predictions.extend(np.argmax(preds, axis=1))
    all_labels.extend(labels.numpy())

# Print the actual distribution
unique_true, counts_true = np.unique(all_labels, return_counts=True)
print(f"True labels in test set: {dict(zip(unique_true, counts_true))}")

unique_preds, counts_preds = np.unique(all_predictions, return_counts=True)
print(f"Model predictions: {dict(zip(unique_preds, counts_preds))}")

# Print classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_predictions, labels=[0, 1], target_names=['Real', 'Manipulated'], zero_division=0))

# Print Accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Accuracy: {accuracy * 100:.2f}%\n")

# Plot confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Manipulated'],
            yticklabels=['Real', 'Manipulated'])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()